# V3 (KCDP) Ablation — Two Views (FULL-TASK and GAP-ONLY)

**Table A (FULL-TASK):** existing scorer, empties included — both-empty problems count as agreement, and κ/AC1 are over all 18 KC slots for every problem.

**Table B (GAP-ONLY):** problems where BOTH human and pred are empty are skipped, then **micro**-averaged precision/recall/F1 over the rest, so the score reflects only the problems that actually carry gaps.

Reuses `utils.metrics` (`evaluate_pair`, `KC_COLUMNS`) and the loader conventions from `ablation_metrics_full.ipynb` without changing them. Both tables report the AvgHuman view (mean of HA-pred and HB-pred) plus an H-H ceiling row (HA vs HB under each table's own definitions).

In [1]:
import json
import sys
from pathlib import Path
from statistics import median

import numpy as np

ROOT = Path("/mnt/d/Projects/kintsugi")
sys.path.insert(0, str(ROOT))

from utils.metrics import evaluate_pair, KC_COLUMNS

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]
CONDITIONS = ["baseline", "no_rules", "reduced", "no_kc"]

HUMAN_DIR = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"
LLM_DIR = ROOT / "results" / "human_validation" / "ablation"
OUT_PATH = ROOT / "ablation_two_views.md"

VALID_KCS = set(KC_COLUMNS)
print("KCs:", len(VALID_KCS), "| Conditions:", CONDITIONS)

KCs: 18 | Conditions: ['baseline', 'no_rules', 'reduced', 'no_kc']


## Loaders (verbatim conventions from `ablation_metrics_full.ipynb`)

pid key = `{sid}_{pid}`; common set = `human_a ∩ human_b ∩ baseline LLM run`.

In [2]:
def find_one(pattern, directory):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matched {pattern} in {directory}")
    return matches[-1]

def normalize_gaps(value):
    if isinstance(value, dict):
        gaps = value.get("gaps", [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {g for g in gaps if g in VALID_KCS}

def load_annotation_file(path):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    sid = str(data.get("studentId", data.get("student_id", "unknown")))
    return sid, {f"{sid}_{pid}": normalize_gaps(v) for pid, v in data.get("annotations", {}).items()}

def merge_files(file_map):
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f"Expected {expected_sid}, found {loaded_sid} in {path.name}")
        merged.update(anns)
    return merged

def load_condition(condition):
    return merge_files({str(sid): LLM_DIR / f"llm_ablation_{condition}_{sid}.json" for sid in STUDENT_IDS})

human_a = merge_files({str(sid): find_one(f"kc_annotations_Pranay Ghuge_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})
human_b = merge_files({str(sid): find_one(f"kc_annotations_Arundhati Das_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})

def common_items():
    """Common set: human_a ∩ human_b ∩ baseline LLM run (same as V3 eval)."""
    ref = load_condition("baseline")
    return sorted(set(human_a) & set(human_b) & set(ref),
                  key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1])))

common = common_items()
print(f"Common problem annotations: {len(common)}")

Common problem annotations: 372


## Table A scorer — FULL-TASK (empties included)

`Problem_F1`, `Jaccard`, `Cohen_kappa`, `Gwet_AC1` come from `evaluate_pair`. Precision/recall use the same empty-set conventions as `problem_f1` (both-empty = 1.0, exactly-one-empty = 0.0), per-problem mean then averaged across the two raters. Human is the reference (`set_a`), pred is the prediction (`set_b`).

In [3]:
def problem_precision(human_set, llm_set):
    if not human_set and not llm_set:
        return 1.0
    if not human_set or not llm_set:
        return 0.0
    tp = len(human_set & llm_set)
    return tp / len(llm_set) if tp else 0.0

def problem_recall(human_set, llm_set):
    if not human_set and not llm_set:
        return 1.0
    if not human_set or not llm_set:
        return 0.0
    tp = len(human_set & llm_set)
    return tp / len(human_set) if tp else 0.0

def mean_pr_vs_human(human, pred, pids):
    precs = [problem_precision(human.get(p, set()), pred.get(p, set())) for p in pids]
    recs = [problem_recall(human.get(p, set()), pred.get(p, set())) for p in pids]
    return float(np.mean(precs)), float(np.mean(recs))

def full_task_view(ref_a, ref_b, pred, pids):
    """AvgHuman full-task metrics: mean of (ref_a vs pred) and (ref_b vs pred)."""
    a = evaluate_pair(ref_a, pred, pids, KC_COLUMNS)
    b = evaluate_pair(ref_b, pred, pids, KC_COLUMNS)
    pa, ra = mean_pr_vs_human(ref_a, pred, pids)
    pb, rb = mean_pr_vs_human(ref_b, pred, pids)
    return {
        "Problem_F1": (a["Problem_F1"] + b["Problem_F1"]) / 2,
        "precision": (pa + pb) / 2,
        "recall": (ra + rb) / 2,
        "Jaccard": (a["Jaccard"] + b["Jaccard"]) / 2,
        "Cohen_kappa": (a["Cohen_kappa"] + b["Cohen_kappa"]) / 2,
        "Gwet_AC1": (a["Gwet_AC1"] + b["Gwet_AC1"]) / 2,
    }

def full_task_pair(ref, pred, pids):
    """Single-pair full-task metrics (used for the H-H ceiling row)."""
    e = evaluate_pair(ref, pred, pids, KC_COLUMNS)
    p, r = mean_pr_vs_human(ref, pred, pids)
    return {
        "Problem_F1": e["Problem_F1"], "precision": p, "recall": r,
        "Jaccard": e["Jaccard"], "Cohen_kappa": e["Cohen_kappa"], "Gwet_AC1": e["Gwet_AC1"],
    }

## Table B scorer — GAP-ONLY (empties removed, micro)

Iterate all `(student, pid)`. **Skip** any problem where BOTH human and pred are empty. Over the kept problems, sum `tp = |h ∩ p|`, `fp = |p − h|`, `fn = |h − p|`, then micro `precision = tp/(tp+fp)`, `recall = tp/(tp+fn)`, `F1 = 2PR/(P+R)`. `exact_match` = fraction of kept problems where `h == p`; `n` = number of kept problems.

In [4]:
def gap_only_micro(ref, pred, pids):
    """Micro precision/recall/F1 over problems where ref or pred is non-empty."""
    tp = fp = fn = kept = exact = 0
    for pid in pids:
        h = ref.get(pid, set())
        p = pred.get(pid, set())
        if not h and not p:
            continue
        kept += 1
        tp += len(h & p)
        fp += len(p - h)
        fn += len(h - p)
        if h == p:
            exact += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "precision": precision, "recall": recall, "F1": f1,
        "exact_match": exact / kept if kept else 0.0, "n": kept,
    }

def gap_only_view(ref_a, ref_b, pred, pids):
    """AvgHuman gap-only: mean of (ref_a vs pred) and (ref_b vs pred)."""
    a = gap_only_micro(ref_a, pred, pids)
    b = gap_only_micro(ref_b, pred, pids)
    return {k: (a[k] + b[k]) / 2 for k in a}

## Score every condition (both views) + H-H ceiling

In [5]:
full_rows, gap_rows = {}, {}
for cond in CONDITIONS:
    pred = load_condition(cond)
    missing = [k for k in common if k not in pred]
    if missing:
        raise ValueError(f"{cond} missing {len(missing)} items, e.g. {missing[:3]}")
    full_rows[cond] = full_task_view(human_a, human_b, pred, common)
    gap_rows[cond] = gap_only_view(human_a, human_b, pred, common)

# H-H ceiling (HA vs HB, same definitions).
full_rows["H-H (ceiling)"] = full_task_pair(human_a, human_b, common)
gap_rows["H-H (ceiling)"] = gap_only_micro(human_a, human_b, common)

ROW_ORDER = CONDITIONS + ["H-H (ceiling)"]

## Sanity checks

`no_kc` gap-only recall vs a single rater should land near ~0.47. If any condition's gap-only `n` differs wildly from the median, that flags a pid mismatch.

In [6]:
warnings = []
cond_ns = [gap_rows[c]["n"] for c in CONDITIONS]
med_n = median(cond_ns)
for c in CONDITIONS:
    n = gap_rows[c]["n"]
    if med_n and (n < 0.5 * med_n or n > 1.5 * med_n):
        warnings.append(f"WARNING: {c} gap-only n={n:.1f} differs wildly from median {med_n:.1f} (pid mismatch?)")

no_kc_single = gap_only_micro(human_a, load_condition("no_kc"), common)["recall"]
print(f"sanity: no_kc gap-only recall vs single rater (HA) = {no_kc_single:.3f} (expected ~0.47)")
for w in warnings:
    print(w)
if not warnings:
    print("sanity: gap-only n consistent across conditions (no pid mismatch).")

sanity: no_kc gap-only recall vs single rater (HA) = 0.467 (expected ~0.47)
sanity: gap-only n consistent across conditions (no pid mismatch).


## Build, print, and save both tables to `ablation_two_views.md`

In [7]:
def table(cols, rows, keys):
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for name in ROW_ORDER:
        vals = [name] + [f"{rows[name][k]:.3f}" if k != "n" else f"{rows[name][k]:.1f}" for k in keys]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

table_a = table(
    ["cond", "Problem_F1", "precision", "recall", "Jaccard", "Cohen_kappa", "Gwet_AC1"],
    full_rows, ["Problem_F1", "precision", "recall", "Jaccard", "Cohen_kappa", "Gwet_AC1"],
)
table_b = table(
    ["cond", "precision", "recall", "F1", "exact_match", "n"],
    gap_rows, ["precision", "recall", "F1", "exact_match", "n"],
)

doc = (
    "# V3 (KCDP) Ablation — Two Views\n\n"
    f"AvgHuman view (mean of HA-pred and HB-pred) over {len(common)} common "
    "problem annotations (human_a ∩ human_b ∩ baseline LLM run), the same pid "
    "list as the main V3 eval. H-H ceiling = HA vs HB under each table's own "
    "definitions.\n\n"
    "## Table A — FULL-TASK (empties included)\n\n"
    "Existing scorer from `utils.metrics`; both-empty problems count as "
    "agreement, and κ/AC1 are over all 18 KC slots for every problem. "
    "Precision/recall use the same empty-set conventions as `problem_f1` "
    "(both-empty = 1.0, exactly-one-empty = 0.0), per-problem mean then "
    "averaged across raters.\n\n"
    f"{table_a}\n\n"
    "## Table B — GAP-ONLY (empties removed, micro)\n\n"
    "Problems where BOTH human and pred are empty are skipped. Over the kept "
    "problems: tp/fp/fn summed across problems, then micro precision/recall/F1. "
    "exact_match = fraction of kept problems where the gap sets are identical; "
    "n = number of kept problems.\n\n"
    f"{table_b}\n"
)

OUT_PATH.write_text(doc)

print(f"Scored over {len(common)} common problem annotations.\n")
print(table_a, "\n")
print(table_b)
print(f"\nWrote {OUT_PATH}")

Scored over 372 common problem annotations.

| cond | Problem_F1 | precision | recall | Jaccard | Cohen_kappa | Gwet_AC1 |
|---|---|---|---|---|---|---|
| baseline | 0.831 | 0.858 | 0.829 | 0.794 | 0.548 | 0.953 |
| no_rules | 0.847 | 0.868 | 0.852 | 0.809 | 0.571 | 0.952 |
| reduced | 0.829 | 0.852 | 0.831 | 0.792 | 0.533 | 0.949 |
| no_kc | 0.820 | 0.859 | 0.809 | 0.782 | 0.496 | 0.950 |
| H-H (ceiling) | 0.885 | 0.906 | 0.881 | 0.851 | 0.669 | 0.963 | 

| cond | precision | recall | F1 | exact_match | n |
|---|---|---|---|---|---|
| baseline | 0.605 | 0.540 | 0.570 | 0.131 | 133.5 |
| no_rules | 0.585 | 0.603 | 0.594 | 0.164 | 134.0 |
| reduced | 0.569 | 0.546 | 0.557 | 0.113 | 133.0 |
| no_kc | 0.590 | 0.464 | 0.519 | 0.126 | 135.0 |
| H-H (ceiling) | 0.718 | 0.658 | 0.687 | 0.252 | 131.0 |

Wrote /mnt/d/Projects/kintsugi/ablation_two_views.md
